In [ ]:
import logging
import mlflow
from datetime import datetime
from pyspark.sql import SparkSession, DataFrame as SparkDataFrame
import pyspark.sql.functions as F
from pyspark.sql.functions import col, lit, pow as spark_pow

logger = logging.getLogger("churn_model")

def _get_or_create_spark() -> SparkSession:
    return SparkSession.builder.getOrCreate()


class ChurnModel(mlflow.pyfunc.PythonModel):
    """MLflow pyfunc model for Customer Churn.

    Runs the full pipeline: ignore_test_customer_ids -> engineer_features -> calculate_churn -> validate_model.
    """

    def __init__(self, params: dict | None = None):
        self.params = params or {}

    def load_context(self, context) -> None:
        pass

    # def predict(self, context, model_input: SparkDataFrame, params: dict | None = None) -> SparkDataFrame: --- IGNORE ---
    def predict(self, context, model_input, params=None):
        spark = _get_or_create_spark()

        if not isinstance(model_input, SparkDataFrame):
            sdf = spark.createDataFrame(model_input)
        else:
            sdf = model_input

        # Step 1: Filter out test customers based on IDs
        sdf = ignore_test_customer_ids(sdf)
        sdf = sdf.filter(~col("test_customer_ids")).drop("test_customer_ids")

        # Step 2: Engineer features for the model
        sdf = engineer_features(sdf)

        # Step 3: Calculate churn probability
        sdf = calculate_churn(sdf)

        # Step 4: Validate model outputs
        validated_sdf = validate_model(sdf)

        return validated_sdf